# CRISPRi screen analysis

This notebook integrates gene-level and individual-sgRNA results from a pooled CRISPRi screen, adds genome annotations, identifies significant changes, and generates tables and visualizations for multiple experimental comparisons. 


In [11]:
import os
from pathlib import Path
import re
from datetime import datetime
import requests
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from matplotlib.ticker import MultipleLocator, NullLocator
from matplotlib_venn import venn3, venn2

## 1. Analysis parameters and input locations

Each comparison is represented by a gene-level summary file and a corresponding sgRNA-level file. Files are paired using the comparison number embedded in their filenames. The genome annotation supplies locus tags, gene names, feature types, and product descriptions.

The significance criteria used in this snapshot are a p-value of at most 0.05, an absolute log2 fold change of at least 1, and at least three sgRNAs assigned to the gene.

In [ ]:
sgrna_dir = Path("path/to/folder/with/sgrna_summary_files")
summary_dir = Path("path/to/folder/with/gene_summary_files")
GENOME_FILE = "path/to/genome_file"

In [13]:
PVAL = 0.05
LOGFC = 1
NB_SGRNA = 3

In [14]:
def extract_number(filename):
    # Extract digits before the extension in the filename
    match = re.search(r'(\d+)\.[^.]+$', filename)
    return match.group(1) if match else None

def extract_condition(filename):
    # Extract condition before the _sgRNA in the filename
    match = re.search(r'([\w\s]+)_sgRNA', filename)
    return match.group(1).strip() if match else None

def collect_datafiles(sgrna_dir, summary_dir):
    """
    Extract digits before the extension for all files in the two given folders and store the info in 
    a dict (digit: file), one for each folder. Extract the numbers common to both folders
    and pair files of each folder that have common numbers in their names.
    Finally, extract for each pair the biological conditions from the first file name
    and return a list of tuples, each containing (condition, (sgrna_file, summary_file))
    """
    # Build dicts: number -> file 
    sgrna_files = {extract_number(f.name): f for f in sgrna_dir.iterdir() if f.is_file()}
    summary_files = {extract_number(f.name): f for f in summary_dir.iterdir() if f.is_file()}
    # Find matching numbers
    common_numbers = set(sgrna_files) & set(summary_files)
    paired_files = [(sgrna_files[n], summary_files[n]) for n in sorted(common_numbers)]

     # Build a dictionary {condition: (file_sgRNA, file_summary)}
    files_to_analyze = {}
    for pair in paired_files:
        condition = extract_condition(pair[0].name)
        files_to_analyze[condition] = pair
    for k, v in files_to_analyze.items():
        print(k,v)
    return files_to_analyze 




In [15]:
def make_genome(file):
    """
    Load a genome annotation CSV file and extract key columns related to gene information, 
    merging values from columns Gene Name and Gene synonyns (sic).

    Parameters:
    ----------
    file : str
        Path to the CSV file containing genome annotation data. The file is expected to have
        columns including 'Locus Tag', 'Feature Type', 'Gene Name', 'Gene synonyns', and 'Product Name'.

    Returns:
    -------
    pandas.DataFrame
        A DataFrame containing the following columns:
        - 'Locus Tag'
        - 'Feature Type'
        - 'Product Name'
        - 'name' : merged values from 'Gene Name' and 'Gene synonyns' (preferring 'Gene Name' if available)
    """
    df = pd.read_csv(file)
    dfc = df[['Locus Tag', 'Feature Type', 'Gene Name', 'Gene synonyns', 'Product Name']].copy()
    dfc['name'] = dfc['Gene Name'].combine_first(dfc['Gene synonyns'])
    dfc = dfc.drop(columns=['Gene synonyns', 'Gene Name'])
    return dfc

In [16]:
def clean_and_merge_df(data_individual_file, data_summary_file, genome, *, pval, logFC):
    """
    Take one file with FC for each sgRNA and one file with FC per gene and merge the data, keeping
    only sgRNAs starting with PA or intergBefore, and adding labelling of significant genes, as
     well as gene name/function from the genome dataframe. 

    Parameters:
    ----------
        - data_individual_file: path to a tsv file
            sgrna_file with data for each sgRNA
        - data_summary_file: path to a tsv file
            Summary file with data for each gene (from MAGIK)
        - genome: DataFrame
            Dataframe with genome information
        - pvalue: float 
            pvalue limit used to assign genes as significant
        - logFC: float
            log2 fold change value used as cutoff to assign genes as significant
    
    Returns:
    -------
    pandas.DataFrame
        A DataFrame containing the following columns:
        
        - 'Gene': Locus tag
        - 'name': gene name
        - 'Feature type': CDS, ...
        - 'Product Name': name of the product of the gene
        - 'Locus Tag': same as 'Gene' for actual PA numbers.
        - 'sgrna': sequence of the sgRNA
        - 'LFC_individual': log2 fold change between the 2 conditions. One for each sgRNA
        - 'nb_of_sgrna': number of sgRNA per locus tag. One for each locus_tag
        - 'P-val': p-value of the median of log2 fold changes of all sgRNAs with more than 10 counts for a given locus tag
        - '-log10(pvalue)': log10 of the P-val
        - 'LFC_summary': median of log2 fold changes of all sgRNAs with more than 10 counts for a given locus tag
        - 'significant': True or False for condition P-Val <= 0.05 and abs(LFC_summary) >= 1
        - 'hue': 'enriched', 'depleted', 'non_significant': separates the previous column on the LFC_summary value 

    """
    # Read the tsv summary file into a df
    df_summary = pd.read_csv(data_summary_file, sep='\t')
    # Add a column with the -log10 of the pvalue
    df_summary["-log10(pvalue)"] = -np.log10(df_summary["P-val"])
    # Add a column with the significance, based on chosen pvalue and logFC cutoff
    df_summary['significant'] = (df_summary['P-val'] <= pval) & (abs(df_summary['LFC']) >= logFC)
    # Add a column for the hue in the volcano plots
    df_summary['hue'] = np.where(df_summary['P-val'] > pval, 'non_significant',
                                 np.where((df_summary['P-val'] <= pval) & (df_summary['LFC'] <= -1), 'depleted', 
                                 np.where((df_summary['P-val'] <= pval) & (df_summary['LFC'] >= 1), 'enriched', 'non_significant')))
    # Read the tsv file containing the data for individual sgRNA
    df = pd.read_csv(data_individual_file, sep='\t')
    # Add the data of the summary file to the individual sgRNA file
    merged_data = df.merge(df_summary, on='Gene', how='outer', suffixes=('_individual', '_summary'))
    # Add a column with the number of sgRNA for each gene
    merged_data['nb_of_sgrna'] = merged_data.groupby('Gene')['Gene'].transform("count")
    # Keep only genes from the column Gene that start with PA or IntergBefore_PA
    cleaned_data = merged_data[(merged_data['Gene'].str.startswith('IntergBefore_PA')) | (merged_data['Gene'].str.startswith('PA'))]
    # Add relevant columns from the genome file
    almost_final_df = cleaned_data.merge(genome, left_on='Gene', right_on='Locus Tag', how='left')
    #Keep only relevant columns
    final_df = almost_final_df[['Gene', 
                                'name', 
                                'Feature Type', 
                                'Product Name', 
                                'Locus Tag', 
                                'sgrna', 
                                'control_count',
                                'treatment_count',
                                'control_mean',
                                'treat_mean',
                                'LFC_individual', 
                                'nb_of_sgrna', 
                                'P-val', 
                                '-log10(pvalue)', 
                                'LFC_summary', 
                                'significant', 
                                'hue']].copy()
    
    # Give the number of na per column in the df
    #for col in final_df.columns:
        #print(col, final_df[col].isna().sum())
 
    return final_df

## 2. Data integration and visualization functions

The helper functions above match related input files, prepare a compact genome annotation, and merge gene-level statistics with individual-guide measurements. Significant genes are labelled as enriched or depleted according to the configured thresholds.

The functions below produce static and interactive volcano plots, display individual-sgRNA fold changes alongside the gene summary value, and calculate overlaps among significant-gene sets.

In [17]:
# Functions to make the figures
import csv
# Choose here the colors: 

colors = {
    "blue":  '#4c72b0',
    "red": '#c44e52',
    "grey": '#8c8c8c',
    "green": '#7c7c7c',
}


def volcano_plot(df, *, title=None, colors=colors):

    fig, ax = plt.subplots(figsize=(5,5))
    sns.scatterplot(data=df, 
                    x='LFC_summary', 
                    y='-log10(pvalue)', 
                    edgecolor='none', 
                    hue='hue', 
                    palette={'depleted': colors["blue"], 'enriched': colors["red"], 'non_significant': colors["grey"]}, 
                    alpha=0.4, 
                    legend=False, 
                    s=20, 
                    ax=ax)
    ax.axhline(-np.log10(0.05), color=colors["green"], linestyle='--')
    ax.axvline(-1, color=colors["green"], linestyle='--')
    ax.axvline(1, color=colors["green"], linestyle='--')
    ax.set_title(f"{title}")
    ax.set_xlabel('log2(fold change)')
    ax.set_ylabel('-log10(pvalue)')
    ax.set_axisbelow(True)
    
    # Manage the grid
    ax.xaxis.set_minor_locator(MultipleLocator(2))   # grid every 1 on x-axis
    ax.grid(True, which='major', alpha=0.3, zorder=0)
    ax.grid(True, which='minor', alpha=0.3, zorder=0)

    # Hide the minor tick marks and labels
    ax.tick_params(which='minor', length=0, labelbottom=False, labelleft=False)

    # Optional: Turn off y-axis minor ticks
    #ax.yaxis.set_minor_locator(NullLocator())

    return fig

def interactive_volcano_plot(df, *, title=None, colors=colors):
   
    # Interactive plot
    fig = px.scatter(
    df, x="LFC_summary", y='-log10(pvalue)', 
    color="hue",
     color_discrete_map={
        'depleted': colors["blue"],
        'enriched': colors["red"], 
        'non_significant': colors["grey"]
    },
    hover_name="Gene",
    hover_data=["Locus Tag", "Feature Type", "Product Name", "name", "nb_of_sgrna"],
    title=f"{title}"
)
    fig.update_traces(marker=dict(size=10), selector=dict(mode='markers'))
    return fig

def get_n_top_significant_genes(df, n, which):
    
    # Use only genes that have at least 3 sgRNA
    df = df[df['nb_of_sgrna'] >= 3]

    # Take only PA genes (start with PA)
    df = df[(df['Gene'].str.startswith('PA'))]

    # Use only previoulsy labelled as significant based on pval and FC 
    df = df[df["significant"] == True]

    # Make a list of the top n genes
    if which == 'top':
        ascending = False
    elif which == 'bottom':
        ascending = True
        
    top_genes = (
        df[['Gene', 'LFC_summary']]
        .drop_duplicates()
        .sort_values('LFC_summary', ascending=ascending)
        .head(n)['Gene']
    )

    # Filter the original dataframe for these genes
    top_df = df[df['Gene'].isin(top_genes)]

    # Sort the dataframe by LFC_summary
    top_df = top_df.sort_values(by='LFC_summary', ascending=False)
   
    return top_df

def plot_significant_genes(df, *, title=None, colors=colors):

    fig, ax = plt.subplots(figsize=(6, 6))
    sns.stripplot(
        data=df, 
        x="Gene", 
        y="LFC_individual", 
        jitter=False, 
        color=colors['blue'],
        alpha=0.2, 
        ax=ax
            )
   
    # Get tick positions from the plot
    xticks = ax.get_xticks()
    xticklabels = [tick.get_text() for tick in ax.get_xticklabels()]
    tick_pos_map = dict(zip(xticklabels, xticks))
    
    # Get the summary LFC
    lfc_summary = df.groupby("Gene")["LFC_summary"].first()
    for gene, lfc in lfc_summary.items():
        if gene in tick_pos_map:
            x_pos = tick_pos_map[gene]
            ax.plot(x_pos, lfc, marker='_', markersize=12, color=colors["red"])

    #ax.set_title("Stripplot with Median")
    #ax.axhspan(-1, 1, color=grey, alpha=0.5)
    ax.tick_params("x", rotation=90)
    #ax.set_xlabel("")
    #ax.set_xticks([])
    ax.set_ylabel('log2(fold change)')
    ax.set_title(f"{title}")
    return fig

def generate_n_plots(datasets, colors=colors):
   
    # Create a grid of volcano plots from a list of dataframes
    fig, axes = plt.subplots(2, 3, figsize=(9, 6), sharex=True, sharey=True)

    for i, ax in enumerate(axes.flat):
        df = datasets[i][1]
        sns.scatterplot(
            data=df,
            x='LFC_summary',
            y='-log10(pvalue)',
            hue='hue', 
            palette={'depleted': colors["blue"], 'enriched': colors["red"], 'non_significant': colors["grey"]}, 
            ax=ax,
            s=20,
            alpha=0.4,
            edgecolor='none',
            legend=False  
        )
        ax.set_box_aspect(1)
        ax.axhline(-np.log10(0.05), linestyle='--', color=colors["green"])
        ax.axvline(-1, linestyle='--', color=colors["green"])
        ax.axvline(1, linestyle='--', color=colors["green"])
        ax.set_title(f'{datasets[i][0]}')
        ax.set_xlabel("log2 Fold Change")
        ax.set_ylabel("-log10(p-value)")
        ax.set_axisbelow(True) 
        
        ax.xaxis.set_major_locator(MultipleLocator(5)) 
        # Manage the grid
        ax.xaxis.set_minor_locator(MultipleLocator(2.5))   # grid every 1 on x-axis
        ax.grid(True, which='major', alpha=0.3, zorder=0)
        ax.grid(True, which='minor', alpha=0.3, zorder=0)
        # Hide the minor tick marks and labels
        ax.tick_params(which='minor', length=0, labelbottom=False, labelleft=False)
        # Optional: Turn off y-axis minor ticks
        #ax.yaxis.set_minor_locator(NullLocator())

    fig.tight_layout()
    return fig

def get_depleted_and_enriched_sets(df):
    # Keep only the real genes (PAxxxx)
    df_PA = df[df['Gene'].str.startswith('PA')]
    depleted = df_PA[df_PA['hue']=='depleted']
    depleted_set = set(depleted['Gene'])

    enriched = df_PA[df_PA['hue']=='enriched']
    enriched_set = set(enriched['Gene'])

    return depleted_set, enriched_set

def analyze_sets(sets_list):

    # Prepare the lists of the depleted sets to save
    in_all = sets_list[0] & sets_list[1] & sets_list[2]
    in_glc_and_succ_not_LB = sets_list[1] & sets_list[2] - sets_list[0]
    in_LB_and_succ_not_glc = sets_list[0] & sets_list[2] - sets_list[1]
    in_LB_and_glc_not_succ = sets_list[0] & sets_list[1] - sets_list[2]
    in_LB_not_glc_not_succ = sets_list[0] - sets_list[1] - sets_list[2]
    in_glc_not_LB_not_succ = sets_list[1] - sets_list[0] - sets_list[2]
    in_succ_not_LB_not_glc = sets_list[2] - sets_list[0] - sets_list[1]
    in_glc_and_succ = sets_list[1] & sets_list[2]
    in_glc_not_succ = sets_list[1] - sets_list[2]
    in_succ_not_glc = sets_list[2] - sets_list[1]


    all_sets = [in_all, 
                in_glc_and_succ_not_LB, 
                in_LB_and_succ_not_glc, 
                in_LB_and_glc_not_succ, 
                in_LB_not_glc_not_succ, 
                in_glc_not_LB_not_succ, 
                in_succ_not_LB_not_glc,
                in_glc_and_succ,
                in_glc_not_succ,
                in_succ_not_glc 
                ]
     
    return all_sets
    

def prep_individual_df(filename, col_lfc, col_hue):
    """Clean the dataframe for each condition to merge all later"""
    df = pd.read_csv(filename, low_memory=False)
    df1 = df.rename(columns={'LFC_summary': col_lfc, 'hue': col_hue})
    df1 = df1[['Gene', 'name', col_lfc, col_hue]]
    df1 = df1[(df1['Gene'].str.startswith('PA'))]
    df1 = df1.drop_duplicates()
    df1.reset_index(drop=True, inplace=True)
    df1.set_index('Gene', inplace=True)
    return df1

## 3. Process one comparison

For a single experimental contrast, this function writes the merged analysis table, creates an interactive volcano plot, and plots the individual guides for the most strongly enriched and depleted genes. One row per sufficiently represented gene is returned for inclusion in the multi-panel volcano plot.

In [18]:

def analyze_one_comparison(condition, sgrna_file, summary_file, genome, output_path, *, pval, logFC, nb_sgrna):

    """
    Generate volcano plots and figures with sgRNA of significant genes for a given comparison

    Parameters: 
        - 
    """

    # Return a dataframe with all sgRNA and their FC, together with the gene FC, the number of sgRNA for the gene and the gene name/function
    df = clean_and_merge_df(sgrna_file, summary_file, genome, pval=pval, logFC=logFC)
    file_csv_name = f"{condition}.csv"
    file_csv_path = output_path / file_csv_name
    df.to_csv(file_csv_path, index=False)

    # This is to check which sgRNAs were not present in the summary file
    #file_csv_name_na = f"{condition}_checkna.csv"
    #file_csv_path_na = output_path / file_csv_name_na
    #df_na = df[df['P-val'].isna()]
    #df_na.to_csv(file_csv_path_na, index=False)

    # Keep only the genes that have at least NB_SGRNA sgRNAs
    df_filtered= df[df['nb_of_sgrna'] >= nb_sgrna]
    
    # Keep only one row per gene to make the volcano plot
    df_for_volcano = df_filtered.drop_duplicates(subset='Gene')
   
    # Create and save the volcano plot
    #file_volcano = f"{condition}_volcano_plot.svg"
    #file_volcano_path = output_path / file_volcano
    #fig = volcano_plot(df_for_volcano, title=condition)
    #fig.savefig(file_volcano_path, dpi=300)
    #plt.close(fig)

    # Create and save the interactive volcano plot
    file_volcano_interactive = f"{condition}_interactive_volcano_plot.html"
    file_volcano_interactive_path = output_path / file_volcano_interactive
    interactive_fig = interactive_volcano_plot(df_for_volcano, title=condition)
    interactive_fig.write_html(file_volcano_interactive_path)
 
    # Create and save the figure that show all sgRNAs for each significant gene
    file_significant = f"{condition}_significant.svg"
    file_significant_path = output_path / file_significant
   
    # Change here how many significant genes you want on the figure (default here is 20)
    top_genes = get_n_top_significant_genes(df, 20, 'top')
    bottom_genes = get_n_top_significant_genes(df, 20, 'bottom')
    top_and_bottom = pd.concat([bottom_genes, top_genes], axis=0, ignore_index=True)
    top_and_bottom = top_and_bottom.sort_values(by='LFC_summary', ascending = False)
   
    fig_sig = plot_significant_genes(top_and_bottom, title=f"{condition}")
    fig_sig.savefig(file_significant_path, dpi=300)
    plt.close(fig_sig)

    # Return the df to make the volcano plot to create later the multi panel volcano plot 
    return df_for_volcano

    
    


In [19]:
def analyze_all(sgrna_dir, summary_dir, genome_file, *, pval, logFC, nb_sgrna):

    # Choose the font for the plots 
    plt.rcParams['svg.fonttype'] = 'none'
    plt.rcParams['pdf.fonttype'] = 42
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.size'] = 10

    # Choose the color palette
    sns.set_palette("muted")  # Options: "deep", "muted", "bright", "pastel", "dark", "colorblind"
    
    # Create output folder 
    base_output_folder = "results"
    # Get today's date in YYYY-MM-DD format
    date_str = datetime.now().strftime('%Y-%m-%d-%H%M%S')

    # Combine folder with date
    output_path = Path(base_output_folder) / date_str
    output_path.mkdir(parents=True, exist_ok=True)

    genome = make_genome(genome_file)

    # Read the files and create a dict: condition: df
    files_to_analyze = collect_datafiles(sgrna_dir, summary_dir)
    df_for_volcano_dict = {}
    for condition, (sgrna_file, summary_file) in files_to_analyze.items():
        df = analyze_one_comparison(condition, sgrna_file, summary_file, genome, output_path, pval=pval, logFC=logFC, nb_sgrna=nb_sgrna)
        df_for_volcano_dict[condition] = df

    # Choose which conditions will be plotted on the figure with mutiple volcano plots
    conditions_to_plot = ["48hLBni vs 48hLB", 
               "48hLBni vs 48hGlc", 
               "48hLBni vs 48hSucc",
               "48hLB vs 48hGlc",
               "48hLB vs 48hSucc",
               "48hGlc vs 48hSucc"
               ]
    
    to_plot = [(x, df_for_volcano_dict[x]) for x in conditions_to_plot if x in df_for_volcano_dict]

    # Save the figure with all volcano plots
    file_all_plots = "volcano_plots_48h.svg"
    file_all_plots_path = output_path / file_all_plots
    fig = generate_n_plots(to_plot)
    fig.savefig(file_all_plots_path, dpi=300)
    plt.close(fig)

    # Prepare the df with all means

    file_LB = "48hLBni vs 48hLB.csv"
    file_glc = "48hLBni vs 48hGlc.csv"
    file_succ = "48hLBni vs 48hSucc.csv"

    file_path_LB = output_path / file_LB
    file_path_glc = output_path / file_glc
    file_path_succ = output_path / file_succ

    df_lb = prep_individual_df(file_path_LB, "LFC_LB", "hue_LB")
    df_glc = prep_individual_df(file_path_glc, "LFC_glc", "hue_glc")
    df_succ = prep_individual_df(file_path_succ, "LFC_succ", "hue_succ")

    all_cond = df_lb.join([df_glc.drop(columns='name'), df_succ.drop(columns='name')], how='inner')

    combos = {
        'in_all': ['LFC_LB', 'LFC_glc', 'LFC_succ'],
        'in_LB_and_glc': ['LFC_LB', 'LFC_glc'],
        'in_LB_and_succ': ['LFC_LB', 'LFC_succ'],
        'in_glc_and_succ': ['LFC_glc', 'LFC_succ']
    }

    for name, cols in combos.items():
        all_cond[f"{name}"] = all_cond[cols].mean(axis=1)

    all_cond_file = output_path / "FC_all.csv"
    all_cond.to_csv(all_cond_file)

    # Choose the conditions for the Venn diagram
    conditions_for_venn = ["48hLBni vs 48hLB", 
               "48hLBni vs 48hGlc", 
               "48hLBni vs 48hSucc"]
    
    conditions_names = ['LB', 'Glucose', 'Succinate']

    for_venn = [(x, df_for_volcano_dict[x]) for x in conditions_for_venn if x in df_for_volcano_dict]

    depleted_sets = []
    enriched_sets = []
    for condition, df in for_venn:
        depleted_set, enriched_set = get_depleted_and_enriched_sets(df)
        depleted_sets.append(depleted_set)
        enriched_sets.append(enriched_set)

    sets_names = ["in_all", 
                  "in_glc_and_succ_not_LB", 
                  "in_LB_and_succ_not_glc", 
                  "in_LB_and_glc_not_succ", 
                  "in_LB_not_glc_not_succ", 
                  "in_glc_not_LB_not_succ", 
                  "in_succ_not_LB_not_glc",
                  "in_glc_and_succ",
                  "in_glc_not_succ",
                  "in_succ_not_glc"]
    
    names_LFC = {"in_all": 'in_all',
                  "in_glc_and_succ_not_LB": 'in_glc_and_succ', 
                  "in_LB_and_succ_not_glc": 'in_LB_and_succ', 
                  "in_LB_and_glc_not_succ":  'in_LB_and_glc', 
                  "in_LB_not_glc_not_succ": 'LFC_LB', 
                  "in_glc_not_LB_not_succ": 'LFC_glc', 
                  "in_succ_not_LB_not_glc": 'LFC_succ',
                  "in_glc_and_succ": 'in_glc_and_succ',
                  "in_glc_not_succ": 'LFC_glc',
                  "in_succ_not_glc": 'LFC_succ'
    }
    
    venn_depleted = analyze_sets(depleted_sets)
    venn_enriched = analyze_sets(enriched_sets)
                    
    # Create and save the enriched venn diagram
    fig, ax = plt.subplots()
    venn3(enriched_sets, set_labels=conditions_names, ax=ax)
    file_venn_enriched = "venn_enriched_all3.svg"
    file_venn_enriched_path = output_path / file_venn_enriched
    fig.savefig(file_venn_enriched_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    # Create and save the depleted venn diagram
    fig, ax = plt.subplots()
    venn3(depleted_sets, set_labels=conditions_names, ax=ax)
    file_venn_depleted = "venn_depleted_all3.svg"
    file_venn_depleted_path = output_path / file_venn_depleted
    fig.savefig(file_venn_depleted_path, dpi=300, bbox_inches='tight')
    plt.close(fig) 

     # Create and save the enriched venn diagram
    fig, ax = plt.subplots()
    venn2(enriched_sets[1:], set_labels=conditions_names[1:], ax=ax)
    file_venn_enriched = "venn_enriched_glc_and_succ.svg"
    file_venn_enriched_path = output_path / file_venn_enriched
    fig.savefig(file_venn_enriched_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    # Create and save the depleted venn diagram for glc and succ
    fig, ax = plt.subplots()
    venn2(depleted_sets[1:], set_labels=conditions_names[1:], ax=ax)
    file_venn_depleted = "venn_depleted_glc_and_succ.svg"
    file_venn_depleted_path = output_path / file_venn_depleted
    fig.savefig(file_venn_depleted_path, dpi=300, bbox_inches='tight')
    plt.close(fig) 


    enrichment_path = output_path / 'depleted_enriched_files'
    enrichment_path.mkdir(parents=True, exist_ok=True)

   
    # Save all lists
    """
    for i, l in enumerate(venn_depleted):
        file_set = f"{sets_names[i]}_depleted.csv"
        file_set_path = enrichment_path / file_set
        with open(file_set_path, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Gene'])
                for gene in l:
                    writer.writerow([gene])

    for i, l in enumerate(venn_enriched):
        file_set = f"{sets_names[i]}_enriched.csv"
        file_set_path = enrichment_path / file_set
        with open(file_set_path, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Gene'])
                for gene in l:
                    writer.writerow([gene])

    """
    for i in range(len(venn_depleted)):

        file_set = f"{sets_names[i]}_both.csv"
        file_set_path = enrichment_path / file_set
    
        both = venn_depleted[i] | venn_enriched[i]

        df = pd.DataFrame({'Gene': list(both)})

        res = df.merge(all_cond, how='left', left_on='Gene', right_on='Gene')
        res1 = res[['Gene', 'name', names_LFC[sets_names[i]]]]
        res1.to_csv(file_set_path)
        
        """
        with open(file_set_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Gene'])
            for gene in both:
                writer.writerow([gene])

        """

In [ ]:
analyze_all(sgrna_dir, summary_dir, GENOME_FILE, pval=PVAL, logFC=LOGFC, nb_sgrna=NB_SGRNA)

## 4. Derive condition-specific gene lists

The main workflow above processes every paired comparison into a timestamped results directory. This additional section extracts one-row-per-gene enriched and depleted tables from selected 48-hour comparisons. It reflects a downstream exploratory step and uses saved result paths from the original analysis environment.

In [ ]:
# Create lists of enriched and depleted genes from one condition

# Files in results/date/
file_glc_succ = "48hGlc vs 48hSucc.csv"
file_lb48 = "48hLBni vs 48hLB.csv"
file_glc48 = "48hLBni vs 48hGlc.csv"
file_succ48 = "48hLBni vs 48hSucc.csv"

file_list = [file_lb48, file_glc48, file_succ48]

def list_enriched_depleted(filepath):

    p = Path(filepath)
    filename = p.stem
    fileparent = p.parent
    destination_filepath = fileparent / filename 

    df = pd.read_csv(filepath, low_memory=False)

    # Use only genes that have at least 3 sgRNA
    df = df[df['nb_of_sgrna'] >= 3]
  
    # Take only PA genes (start with PA)
    df = df[(df['Gene'].str.startswith('PA'))]
    
    # Keep only one row per gene 
    df = df.drop_duplicates(subset='Gene')
   
    # Drop sgrna columns 
    df = df.drop(columns=['sgrna', 'LFC_individual'])

    # Get the enriched and depleted lists 
    df_enriched = df[df['hue'] == 'enriched']
    df_depleted = df[df['hue'] == 'depleted']

    # Save the lists
    destination_filepath_enriched = f"{destination_filepath}_enriched.csv"
    df_enriched.to_csv(destination_filepath_enriched, index=False)

    destination_filepath_depleted = f"{destination_filepath}_depleted.csv"
    df_depleted.to_csv(destination_filepath_depleted, index=False)
   
    



In [ ]:

for file in file_list:
    list_enriched_depleted(file)

### Examples of figures

![volcano_plot](figures/volcano_plot.svg)

![significant](figures/significant.svg)